In [10]:
import _winapi
import os
import shutil
from pathlib import Path

import pandas as pd

In [5]:
def anonymize(dataset: str) -> list:
    
    df = pd.read_csv(dataset)
    nb_lines = df.shape[0]
    
    # histo from desease
    
    df["NAME"] = ""
    df["NAME"] = [f"{i:04d}.png" for i in range(nb_lines)]
    
    df.to_csv(dataset, index=False)

In [18]:
def create_dataset(name: str, output_path: str, output_path_csv: str | None = None, quantity: float = 1.0) -> pd.DataFrame:
    """
        params:
        - Quantity: Quantity (in normalize pourcentage) of the dataset filename (input file)
        takes to create the real dataset.
        Create a csv of the subdataset if the quantity < 1
    """
    
    assert quantity > 0
    assert len(output_path) > 0 and len(name) > 0
    
    filename = Path(name)
    output = Path(output_path)
    
    df: pd.DataFrame = pd.read_csv(filename)
    n_rows = 0
    
    if quantity <= 1:
        n_rows = int(len(df) * quantity)
        n_rows += n_rows % 2
    else:
        n_rows = int(quantity)
    
    df = df.iloc[:n_rows]
    
    left_path =  output / "left"
    right_path = output / "right"
    
    left_path.mkdir(parents=True, exist_ok=True)
    right_path.mkdir(parents=True, exist_ok=True)
    
    left_df: pd.DataFrame = df.loc[df["SIDE"] == "L"]
    right_df: pd.DataFrame = df.loc[df["SIDE"] == "R"]
    
    def _create_class(class_df: pd.DataFrame, path: Path):
        for _, row in class_df.iterrows():
        
            source = row["SOURCE"]
            name = row["NAME"]
            
            source = source.strip('"')
            name = name.strip()

            shutil.copy(source, path / name)
        
        
    _create_class(left_df, left_path)
    _create_class(right_df, right_path)
    
    if quantity != 1.0 or output_path_csv is not None:
        basename = Path(name).stem
        
        output_csv = Path(output_path_csv, f"{basename}_{quantity}.csv")
        
        df.to_csv(output_csv)
    
    return df

def link_dir(target, link):
    if Path(link).exists():
        return
    
    _winapi.CreateJunction(os.path.abspath(target), os.path.abspath(link))

def add_symlink(path, symlink):
    os.symlink(path, symlink)

In [20]:
anonymize("source/csv/sources_kfold_train.csv")
create_dataset("source/csv/sources_kfold_train.csv", "main/train")

anonymize("source/csv/sources_kfold_test.csv")
create_dataset("source/csv/sources_kfold_test.csv", "main/test")

,SOURCE,PATIENT,TYPE,SIDE,PATHO,NAME
0,"""Y:\260113\260113_LAC_L_1\260113_LAC_L_1_HD\pn...",LAC,retine,L,NaN,0000.png
1,"""Y:\260113\260113_LAC_R_1\260113_LAC_R_1_HD\pn...",LAC,retine,R,NaN,0001.png
2,"""Y:\260119\260119_TRE_L_1\260119_TRE_L_1_HD\pn...",TRE,retine,L,NaN,0002.png
3,"""Y:\260119\260119_TRE_R_1\260119_TRE_R_1_HD\pn...",TRE,retine,R,NaN,0003.png
4,"""Y:\260120\260120_YOS_L_1\260120_YOS_L_1_HD\pn...",YOS,retine,L,NaN,0004.png
...,...,...,...,...,...,...
79,"""Y:\251110\251110_SAF_R_1\251110_SAF_R_1_HD\pn...",SAF,retine,R,NaN,0079.png
80,"""Y:\251110\251110_WAD_L_1\251110_WAD_L_1_HD\pn...",WAD,retine,L,NaN,0080.png
81,"""Y:\251110\251110_WAD_R_1\251110_WAD_R_1_HD\pn...",WAD,retine,R,NaN,0081.png
82,"""Y:\251110\251110_WIM_L_2\251110_WIM_L_2_HD\pn...",WIM,retine,L,NaN,0082.png


## Create multiple dataset for benchmark

In [14]:
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.2/train", "source/csv", quantity=0.2)
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.4/train", "source/csv", quantity=0.4)
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.6/train", "source/csv", quantity=0.6)
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.8/train", "source/csv", quantity=0.8)

link_dir("main/test", "benchmark/kfold_0.2/test")
link_dir("main/test", "benchmark/kfold_0.4/test")
link_dir("main/test", "benchmark/kfold_0.6/test")
link_dir("main/test", "benchmark/kfold_0.8/test")